# White noise, seeds and realizations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/experimental/random/noise_and_seeds.ipynb)

One convention for randomness, exercised end to end:

- a **`WhiteNoise` block** is white noise of intensity `psd` held over a `sample_period`; its seed, period and intensity are ordinary params, and its signal is a pure function of `(t, params)`;
- **`realize(key)`** gives every random block of a diagram its own fresh seed under one key, and `seed=None` is the mean;
- a diagram holding a noise block **picks its solver by itself**, and the analysis verbs read it **at the mean**.

What to look for while running it: the variance of a first-order plant matches the Lyapunov value for two different sample periods; the simulator reports fixed-step RK4 on a divisor of the period; the three warnings say what they should; the noisy loop linearizes to the noise-free matrices.

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

from minilink import (
    DiagramSystem,
    ImpedanceController,
    LowPassFilter,
    PendulumWithNoisePort,
    Step,
    WhiteNoise,
)
from minilink.analysis import find_equilibrium, jacobian, linearize_matrices
from minilink.simulation import Simulator

## 1. One block

`WhiteNoise(p, *, psd, sample_period, seed, hold)`. The intensity `psd` is the two-sided spectral density, in units of the signal squared times seconds. Each sample is drawn as $w_k \sim \mathcal{N}(0, \text{psd} / \Delta)$ and held over the period $\Delta$, so the plant feels the same intensity whatever the period. Everything the block reads is in its `params`.

In [ ]:
w = WhiteNoise(1, psd=0.01, sample_period=0.01, seed=1)
w.params

In [ ]:
w.show_signal(t0=0.0, tf=2.0);

## 2. Edit the params, no refresh

A longer period holds fewer, smaller samples for the same intensity; a new seed is a new realization. The block draws from `(t, params)` at every call, so an edit is live at once.

In [ ]:
w.params["sample_period"] = 0.1
w.show_signal(t0=0.0, tf=2.0);

In [ ]:
w.params["seed"] = 2
w.show_signal(t0=0.0, tf=2.0);

## 3. The mean, and a noise switched off

`seed=None` is the mean of the law, zero; so is a zero intensity.

In [ ]:
empty = np.zeros(0)
WhiteNoise(1, seed=None).h(empty, empty, 0.3), WhiteNoise(1, psd=0.0).h(empty, empty, 0.3)

## 4. The physics does not change with the period

For $\dot{x} = -a x + w$ with $w$ white of intensity $W$, the stationary variance solves the Lyapunov equation $0 = -2aP + W$, so $P = W / 2a$. The held train reproduces it at the sample instants for a period of 10 ms and of 100 ms alike, because each sample carries the covariance $W / \Delta$.

In [ ]:
from minilink.dynamics.abstraction.state_space import LTISystem

a, W = 1.0, 1.0
P = W / (2 * a)

variance = {}
for period in (0.01, 0.1):
    plant = LTISystem(np.array([[-a]]), np.array([[1.0]]))
    noise = WhiteNoise(1, psd=W, sample_period=period)
    loop = noise >> plant
    samples = []
    for seed in range(8):
        noise.params["seed"] = seed
        traj = loop.compute_trajectory(tf=40.0, dt=period, solver="rk4_fixedsteps", verbose=False)
        samples.append(traj.x[0, int(5.0 / period) :])  # after five time constants
    variance[period] = np.var(np.concatenate(samples))

np.array([[period, variance[period], P] for period in variance])  # period, measured, Lyapunov

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar([str(p) for p in variance], list(variance.values()), width=0.5, label="held train, sample instants")
ax.axhline(P, color="k", linestyle="--", label="W / 2a")
ax.set_xlabel("sample period [s]")
ax.set_ylabel("Var[x]")
ax.legend()
plt.show()

## 5. A noisy closed loop

Process noise on the plant's `w` port, measurement noise on its `v` port, each block with its own seed. The block publishes its period as a solver hint: with no solver named, the simulator takes fixed-step RK4 on a divisor of the period, one step per sample here since the period is the resolution you chose. A named `solver` or `dt` still wins: `dt=0.001` gives ten steps per sample for a long, precise run.

In [ ]:
plant = PendulumWithNoisePort()
plant.params["m"] = 1.0
plant.params["l"] = 5.0
plant.x0[0] = 2.0

process_noise = WhiteNoise(1, psd=0.01, seed=1)
measurement_noise = WhiteNoise(1, psd=0.001, seed=2)

loop = DiagramSystem()
loop.add_subsystem(Step(final_value=1.0, step_time=5.0), "step")
loop.add_subsystem(ImpedanceController(Kp=100.0, Kd=50.0), "controller")
loop.add_subsystem(plant, "plant")
loop.add_subsystem(process_noise, "process_noise")
loop.add_subsystem(measurement_noise, "measurement_noise")
loop.connect("step", "y", "controller", "r")
loop.connect("controller", "u", "plant", "u")
loop.connect("plant", "y", "controller", "y")
loop.connect("process_noise", "y", "plant", "w")
loop.connect("measurement_noise", "y", "plant", "v")
loop.solver_info

In [ ]:
loop.plot_diagram()

In [ ]:
traj = loop.compute_trajectory(tf=10.0)
loop.plot_trajectory()

In [ ]:
sim = Simulator(loop, tf=10.0, verbose=False)
sim.solver_mode, sim.dt

## 6. The solver warnings

Adaptive stepping fights the jump at every sample, and a step that does not divide the period reads one sample in several. Both are announced.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    loop.compute_trajectory(tf=2.0, solver="scipy", verbose=False)
[str(c.message) for c in caught]

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    loop.compute_trajectory(tf=2.0, dt=0.003, solver="rk4_fixedsteps", verbose=False)
[str(c.message) for c in caught]

## 7. `realize`: one key per experiment

`realize(None)` returns the params with every random block at its mean; `realize(key)` returns them with a fresh seed in each random block, derived from the key and the block's id, so adding a block reseeds no other. The result nests like `params` and can be assigned to it. A Monte Carlo run is a loop over keys.

In [ ]:
loop.realize(None)["process_noise"], loop.realize(None)["measurement_noise"]

In [ ]:
loop.realize(3)["process_noise"], loop.realize(3)["measurement_noise"]

In [ ]:
loop.x0

In [ ]:
angles = []
loop.subsystems["plant"].x0[0] = 0.0

for key in range(8):
    # print(loop.params)
    loop.params = loop.realize(key)
    print(loop.params)
    traj = loop.compute_trajectory(tf=20.0, dt=0.01, verbose=False)
    angles.append(traj.x[0])
loop.params = loop.realize(None)  # back to the mean

fig, ax = plt.subplots(figsize=(7, 3))
for angle in angles:
    ax.plot(traj.t, angle, alpha=0.6)
ax.set_xlabel("t [s]")
ax.set_ylabel("theta [rad]")
ax.set_title("eight realizations of the loop")
plt.show()

## 8. A shared seed is announced

Two blocks with one seed draw one signal. Compiling the diagram says so.

In [ ]:
twins = DiagramSystem()
twins.add_subsystem(PendulumWithNoisePort(), "plant")
twins.add_subsystem(WhiteNoise(), "process_noise")
twins.add_subsystem(WhiteNoise(), "measurement_noise")
twins.connect("process_noise", "y", "plant", "w")
twins.connect("measurement_noise", "y", "plant", "v")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    twins.compile(backend="numpy", verbose=False)
[str(c.message) for c in caught]

## 9. Analysis at the mean

With `params=None`, every analysis verb reads a random block at its mean, so a noisy loop is linearized at $E[w] = 0$ and its equilibrium found there, while the block's `psd` stays readable for a filter design.

In [ ]:
quiet = DiagramSystem()
quiet.add_subsystem(Step(final_value=1.0, step_time=10.0), "step")
quiet.add_subsystem(ImpedanceController(Kp=100.0, Kd=50.0), "controller")
quiet.add_subsystem(PendulumWithNoisePort(), "plant")
quiet.connect("step", "y", "controller", "r")
quiet.connect("controller", "u", "plant", "u")
quiet.connect("plant", "y", "controller", "y")
quiet.subsystems["plant"].params.update(m=1.0, l=5.0)
quiet.x0 = loop.x0.copy()

A_noisy = jacobian(loop, "f", "x", method="fd")
A_quiet = jacobian(quiet, "f", "x", method="fd")
A_noisy, np.allclose(A_noisy, A_quiet)

In [ ]:
find_equilibrium(loop, loop.x0)

## 10. The numbers a Kalman design reads

The continuous filter reads $Q = B_w W B_w^T$ and $R = D_v V D_v^T$ from the two intensities; a discrete filter at a period $\Delta$ reads $Q_d = B_w W B_w^T \Delta$ and $R_d = D_v V D_v^T / \Delta$. Times $\Delta$ on the state, over $\Delta$ on the measurement.

In [ ]:
plant = PendulumWithNoisePort()
A, B, C, D = linearize_matrices(plant, wrt="u")
B_w = linearize_matrices(plant, wrt="w")[1]
D_v = linearize_matrices(plant, wrt="v")[3]

Q = B_w @ np.diag(process_noise.psd) @ B_w.T
R = D_v @ np.diag(measurement_noise.psd) @ D_v.T
Q, R

## 11. Colored noise is white noise through a filter

A first-order filter with time constant $\tau$ turns white noise of intensity $W$ into a signal of variance $W / 2\tau$ and correlation time $\tau$. The filter starts at rest, so the first $\tau$ seconds are quiet; the plotted state is the realization's, the output is $C x$.

In [ ]:
tau = 0.5
shaping = LowPassFilter(cutoff_hz=1.0 / (2.0 * np.pi * tau))
wind = WhiteNoise(1, psd=1.0, seed=4) >> shaping
traj = wind.compute_trajectory(tf=40.0)
wind.plot_diagram()


In [ ]:
wind.plot_trajectory()

In [ ]:
z = shaping.C() @ traj.x  # the filter's output; its realization's state is scaled by C
np.var(z[0, traj.t > 2.0]), 1.0 / (2.0 * tau)  # measured, W / 2τ

## 12. Noise the stepped tools cannot see

The reinforcement-learning environment and the Monte Carlo evaluator step the plant with the true state fed to the law; a disturbance that only reaches the output map changes nothing there, and a random block inside the plant keeps one seed across trials. Both are announced. Simulating the whole closed-loop diagram for every trial is the next step of the convention.

In [ ]:
from minilink.core.costs import QuadraticCost
from minilink.core.distributions import Gaussian, Uniform
from minilink.planning import StochasticPlanningProblem
from minilink.planning.reinforcement_learning.environment import RolloutEnvironment

problem = StochasticPlanningProblem(
    PendulumWithNoisePort(),
    cost=QuadraticCost.from_system(PendulumWithNoisePort()),
    tf=2.0,
    x0_distribution=Uniform([-0.5, -0.5], [0.5, 0.5]),
    disturbances={"v": Gaussian(0.0, 0.1)},
)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    RolloutEnvironment(problem, dt=0.05, backend="numpy")
[str(c.message) for c in caught]

## 13. The same signal on JAX

One cipher on both backends: a seed is the same signal wherever the diagram compiles. The two rollouts differ only by the backends' rounding.

In [ ]:
from minilink.core.backends import jax_installed

if jax_installed():
    on_numpy = loop.compute_trajectory(tf=5.0, dt=0.01, compile_backend="numpy", verbose=False)
    on_jax = loop.compute_trajectory(tf=5.0, dt=0.01, compile_backend="jax", verbose=False)
    difference = float(np.max(np.abs(on_numpy.x - np.asarray(on_jax.x))))
else:
    difference = "jax is not installed"
difference

## What comes next

Sampled noise of any law (`NoiseSource`), a planning problem that realizes its start, its parameters and its disturbances under one key, and a Monte Carlo evaluator that simulates the whole closed-loop diagram on a fixed set of realizations, so any controller and any noise block go through the one simulation path.